# 02. 피처 실험

- 담당: **② 홍시윤 (데이터·피처)** (이 파일은 담당자만 수정)
- 기록 파일: `experiments/log_H_data.csv`

### 할 일
- [ ] 기본 피처(WT=0, 변이=1)와 비교할 새 피처 만들기 (변이 개수, 변이 종류 구분 등)
- [ ] `run_cv()`로 기본 피처보다 오르는지 확인 → 오르면 `src/features.py`로 옮기기
- [ ] 피처 선택은 반드시 Pipeline 안에서 (fold 안에서만 fit)
- [ ] **화요일 12시 피처 확정**

> **규칙:** 전처리는 Pipeline 안에 / test는 보지 않기 / 점수는 `run_cv()`로 / 결과는 `log_result()`로 기록 / 자세한 규칙은 `docs/coding_rules.md`

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [2]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/ah07-cancer-subtype/data


In [3]:
import platform
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import basic_transformer, binarize_mutations
from src.features import mutation_count
from src.validation import get_folds, run_cv, balanced_weights
from src.evaluation import macro_f1, per_class_f1, log_result
from src.prediction import make_submission

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

OS: macOS-15.7.3-arm64-arm-64bit-Mach-O | python 3.14.8 | pandas 3.1.1 | scikit-learn 1.9.1


## 1. 데이터

In [4]:
train, test = load_data(DATA_DIR)   # test는 형식 확인과 최종 추론에만
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                 # 팀 공통 fold
print("train", train.shape, "| test", test.shape, "| classes", len(le.classes_))

train (6201, 4386) | test (2546, 4385) | classes 26


## 실험 예시 (복사해서 바꿔 쓰세요)
1. Pipeline에 피처 변환과 모델을 넣고
2. `run_cv()`로 점수를 내고
3. `log_result()`로 기록해요.

In [5]:
# from xgboost import XGBClassifier
# PARAMS = dict(n_estimators=200, learning_rate=0.1, max_depth=6, tree_method="hist", random_state=SEED, n_jobs=-1)
# pipe = Pipeline([("features", basic_transformer()), ("model", XGBClassifier(**PARAMS))])
# cv = run_cv(pipe, X, y, folds=folds)
# per_class_f1(y, cv["oof"], le.classes_).head(10)
# log_result(owner="H_data", notebook="02_features.ipynb", experiment="실험 이름",
#            features="basic(WT=0,변이=1)", model="XGBoost", params=PARAMS, cv=cv, notes="")

=============================공통 코드 끝=============================

## 2. 피처 구성 방법별 실험

### A. 원본 인코딩

In [38]:
# 상의 후 진행예정

### B. 이진 인코딩

In [6]:
# B. 이진 인코딩 결과 확인 (WT=0, 변이=1)

X_B = binarize_mutations(X)

print("원본 데이터 크기:", X.shape)
print("이진 인코딩 데이터 크기:", X_B.shape)
print("변환 후 고유값:", np.unique(X_B.to_numpy()))

display(X_B.iloc[:5, :10])

원본 데이터 크기: (6201, 4384)
이진 인코딩 데이터 크기: (6201, 4384)
변환 후 고유값: [0 1]


,A2M,AAAS,AADAT,AARS1,ABAT,ABCA1,ABCA2,ABCA3,ABCA4,ABCA5
0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0


### C. 이진 인코딩 + 변이 개수

In [7]:
# C. 이진 인코딩 + 환자별 변이 유전자 개수 피처 생성

# B 방식의 이진 인코딩 결과 복사
X_C = X_B.copy()

# 환자별 변이 유전자 개수 추가
X_C["n_mutated_genes"] = mutation_count(X)

# 결과 확인
print("B 방식 데이터 크기:", X_B.shape)
print("C 방식 데이터 크기:", X_C.shape)

print("환자별로 변이가 기록된 유전자 수")
display(X_C[["n_mutated_genes"]].head(10))

B 방식 데이터 크기: (6201, 4384)
C 방식 데이터 크기: (6201, 4385)
환자별로 변이가 기록된 유전자 수


,n_mutated_genes
0,25
1,17
2,119
3,7
4,34
5,270
6,4
7,1
8,9
9,20


In [8]:
# C 방식의 변이 유전자 개수 피처 검증

count_check = X_B.sum(axis=1)

print(
    "변이 개수 일치 여부:",
    (count_check == X_C["n_mutated_genes"]).all()
)

print(
    "결측치 개수:",
    X_C["n_mutated_genes"].isna().sum()
)

변이 개수 일치 여부: True
결측치 개수: 0


### D. 이진 인코딩 + 변이 개수 + 변이 유형별 개수

In [9]:
# D. 변이 유형별 개수 피처 생성을 위한 원본 변이값 확인

# WT와 결측치를 제외한 실제 변이 기록만 추출
mutation_values = X.stack()
mutation_values = mutation_values[mutation_values != "WT"]

print("전체 변이 기록 셀 수:", len(mutation_values))
print("서로 다른 변이 표기 수:", mutation_values.nunique())

print("\n가장 자주 등장하는 변이 표기 TOP 30")
display(mutation_values.value_counts().head(30))

print("\n변이 표기 예시 30개")
print(mutation_values.drop_duplicates().head(30).tolist())

전체 변이 기록 셀 수: 218893
서로 다른 변이 표기 수: 118846

가장 자주 등장하는 변이 표기 TOP 30


R132H     349
V600E     305
E545K     135
H1047R    128
R273C      82
E542K      68
R175H      66
R273H      60
R132C      59
R248Q      49
R213*      45
R248W      45
L42L       39
R282W      34
R196*      32
Y220C      32
K16fs      31
R80*       31
L35L       28
L100L      27
A2A        25
Q61R       25
Q269Q      25
H179R      25
L45L       24
P423P      24
R306*      24
G245S      24
E110K      24
L159L      23
Name: count, dtype: int64


변이 표기 예시 30개
['D623D', 'F736C', 'P117R', 'R305S', 'N23K', 'L192V', 'G592G', 'K277K', 'A699T', 'L111L', 'I136M', 'A219T', 'E237*', 'A746D', 'D465D', 'N1445D', 'T218T', 'G764G', 'E660Q', 'E291A', 'I382M', 'V412A', 'P222R', 'R104H', 'P17R', 'T1115M', 'L1854fs', 'R36H', 'R69Q', 'A372T']


In [11]:
# D. 한 유전자 셀에 여러 변이 표기가 기록된 경우 확인

# WT와 결측치를 제외한 변이 기록 추출
mutation_values = X.stack()
mutation_values = mutation_values[mutation_values != "WT"]

# 공백 기준으로 나뉘는 표기 개수 계산
token_counts = mutation_values.astype(str).str.split().str.len()

print("전체 변이 기록 셀 수:", len(mutation_values))
print("표기가 1개인 셀 수:", (token_counts == 1).sum())
print("표기가 2개 이상인 셀 수:", (token_counts >= 2).sum())
print("한 셀의 최대 표기 개수:", token_counts.max())

print("\n표기 개수별 셀 수")
display(token_counts.value_counts().sort_index().head(64))

print("\n표기가 2개 이상인 셀 예시 20개")
display(mutation_values[token_counts >= 2].head(20))

전체 변이 기록 셀 수: 218893
표기가 1개인 셀 수: 196867
표기가 2개 이상인 셀 수: 22026
한 셀의 최대 표기 개수: 64

표기 개수별 셀 수


1     196867
2      16261
3       3023
4       1399
5        382
6        374
7         79
8        159
9         25
10        82
11        16
12        63
13         6
14        40
15         5
16        28
17         2
18        19
19         1
20        19
21         1
22        12
24         8
26         6
28         3
30         4
32         2
36         2
38         1
39         1
44         1
60         1
64         1
Name: count, dtype: int64


표기가 2개 이상인 셀 예시 20개


2  ITGB5        Q369* I368N
   LAMB3        S622S G827R
   MYH4        E412K R1800C
   RYR1       D3546V G3739G
   TNC          A368V E646K
5  ADAMTS1      C521Y R258R
   ADCY9        G273R V155A
   ALDH3A1       F336L I71T
   ANO1        I758S P700fs
   BCAN         R107H E468E
   DIAPH3        Q924* P44P
   HDAC9         L99L N825N
   IRF2         K64T N167fs
   JAK2       Y435fs N193fs
   MLF1           H22H G49W
   MMP2         G179G A408A
   PAOX         S269S T286T
   PIAS3       V274M L226fs
   REV3L      Y2576F F1486F
   RHOBTB3      S372S T480T
dtype: str

In [14]:
# D. 전체 변이 표기 분리 및 유형별 분류 결과 확인

# 공백 기준으로 개별 변이 표기 분리
tokens = mutation_values.astype(str).str.split().explode()

# 변이 유형 분류
is_nonsense = tokens.str.fullmatch(r"[A-Z]\d+\*")
is_frameshift = tokens.str.contains("fs", case=False, regex=False)

is_substitution = tokens.str.fullmatch(r"[A-Z]\d+[A-Z]")

is_synonymous = (
    is_substitution
    & (tokens.str[0] == tokens.str[-1])
)

is_missense = is_substitution & ~is_synonymous

# 기타/미분류
is_other = ~(
    is_nonsense
    | is_frameshift
    | is_synonymous
    | is_missense
)

# 분류 결과 확인
print("전체 개별 변이 표기 수:", len(tokens))
print("미스센스:", is_missense.sum())
print("동의 변이:", is_synonymous.sum())
print("넌센스:", is_nonsense.sum())
print("프레임시프트 후보:", is_frameshift.sum())
print("기타/미분류:", is_other.sum())

print("\n분류 중복 개수:",
      (is_frameshift & (is_missense | is_synonymous | is_nonsense)).sum())

print("\n기타/미분류 표기 TOP 30")
display(tokens[is_other].value_counts().head(30))

전체 개별 변이 표기 수: 255164
미스센스: 164740
동의 변이: 66883
넌센스: 13289
프레임시프트 후보: 9914
기타/미분류: 338

분류 중복 개수: 0

기타/미분류 표기 TOP 30


*261*             3
*1232*            3
*459*             3
468_469LG>F*      2
312_313QY>HH      2
82_83DA>EI        2
139_140MG>I*      2
292_293DP>ES      2
127_128KL>KL      2
929_930LL>LL      2
435_436EA>DS      2
*399*             2
*271*             2
711_712FL>FL      2
*220*             2
594_595ME>IK      2
*902*             2
*626*             2
*235*             2
1105_1106PL>PL    2
187_188PL>PL      2
1441_1442AL>AL    2
*837*             2
2069_2070WA>*T    2
*455*             2
*194*             2
*141*             2
*924*             2
*335*             2
264_265MG>I*      2
Name: count, dtype: int64

#### 대표적인 기타 미분류의 변이 표기 
| 표기 | 해석 |
| :--- | :--- |
| `*261*` | 종결 코돈이 유지된 것으로 보이는 표기 |
| `312_313Q>HH` | 두 아미노산이 함께 치환된 표기 |
| `127_128KL>KL` | 두 아미노산이 동일하게 유지된 표기 |
| `468_469LG>F*` | 복합 아미노산 변화 및 종결 코돈 포함 |

#### 추가 피처명
| 변수명 | 설명 |
| :--- | :--- |
| `n_missense` | 단일 아미노산 치환 표기 수 |
| `n_synonymous` | 단일 아미노산 동의 변이 표기 수 |
| `n_nonsense` | 단일 아미노산이 종결 코돈으로 바뀐 표기 수 |
| `n_frameshift` | `fs`가 포함된 프레임시프트 후보 표기 수 |
| `n_other` | 나머지 복합 변이 및 미분류 표기 수 |

In [15]:
# D. 환자별 변이 유형 개수 피처 생성 함수

import re
import pandas as pd

def mutation_type_counts(X):
    # 변이 유형별 개수를 저장할 데이터프레임
    feature_names = [
        "n_missense",
        "n_synonymous",
        "n_nonsense",
        "n_frameshift",
        "n_other",
    ]

    result = pd.DataFrame(
        0,
        index=X.index,
        columns=feature_names,
        dtype="int32"
    )

    # WT와 결측치를 제외한 변이 기록 추출
    mutations = X.stack()
    mutations = mutations[mutations != "WT"]

    # 한 셀에 여러 변이가 있으면 공백 기준으로 분리
    tokens = mutations.astype(str).str.split().explode()

    # 개별 변이 표기를 유형별로 분류
    def classify(token):
        if "fs" in token.lower():
            return "n_frameshift"

        if re.fullmatch(r"[A-Z]\d+\*", token):
            return "n_nonsense"

        if re.fullmatch(r"[A-Z]\d+[A-Z]", token):
            if token[0] == token[-1]:
                return "n_synonymous"
            return "n_missense"

        return "n_other"

    types = tokens.map(classify)

    # 환자별 변이 유형 개수 집계
    counts = pd.crosstab(
        tokens.index.get_level_values(0),
        types
    )

    result = counts.reindex(
        index=X.index,
        columns=feature_names,
        fill_value=0
    ).astype("int32")

    return result

In [16]:
# D. 이진 인코딩 + 변이 개수 + 변이 유형별 개수 생성

type_counts = mutation_type_counts(X)

X_D = pd.concat([X_C, type_counts], axis=1)

print("C 방식 데이터 크기:", X_C.shape)
print("D 방식 데이터 크기:", X_D.shape)

display(type_counts.head(10))

C 방식 데이터 크기: (6201, 4385)
D 방식 데이터 크기: (6201, 4390)


col_0,n_missense,n_synonymous,n_nonsense,n_frameshift,n_other
0,17,7,1,0,0
1,9,5,0,3,0
2,79,39,6,0,0
3,3,3,1,0,0
4,23,11,0,0,0
5,164,81,11,31,0
6,2,1,0,1,0
7,0,1,0,0,0
8,7,2,0,0,0
9,12,6,2,0,0


In [17]:
# D. 변이 유형별 개수 피처의 합계 및 결측치 검증

# 환자별 변이 유형 개수 합계
total_type_counts = type_counts.sum(axis=1)

print("전체 변이 표기 수:", len(tokens))
print("유형별 개수 총합:", total_type_counts.sum())

print("전체 합계 일치 여부:",
      total_type_counts.sum() == len(tokens))

print("결측치 개수:", type_counts.isna().sum().sum())

print("변이 유전자 수보다 변이 표기 수가 적은 환자:",
      (total_type_counts < X_C["n_mutated_genes"]).sum())

print("\n변이 유형별 전체 개수")
display(type_counts.sum())

전체 변이 표기 수: 255164
유형별 개수 총합: 255164
전체 합계 일치 여부: True
결측치 개수: 0
변이 유전자 수보다 변이 표기 수가 적은 환자: 0

변이 유형별 전체 개수


col_0
n_missense      164740
n_synonymous     66883
n_nonsense       13289
n_frameshift      9914
n_other            338
dtype: int64

## 3. 피처 구성별 모델 성능 비교

### 3-1. Pipeline 피처 변환기 구성 및 검증

In [18]:
# B·C·D 피처 구성을 위한 Pipeline 변환 함수

from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

def make_features_B(X):
    return binarize_mutations(X)

def make_features_C(X):
    X_binary = binarize_mutations(X)

    return pd.concat([
        X_binary,
        mutation_count(X)
    ], axis=1)

def make_features_D(X):
    X_binary = binarize_mutations(X)

    return pd.concat([
        X_binary,
        mutation_count(X),
        mutation_type_counts(X)
    ], axis=1)

# Pipeline에서 사용할 변환기
transformer_B = FunctionTransformer(make_features_B)
transformer_C = FunctionTransformer(make_features_C)
transformer_D = FunctionTransformer(make_features_D)

# 기존 결과와 동일하게 생성되는지 확인
print("B 일치:", transformer_B.transform(X).equals(X_B))
print("C 일치:", transformer_C.transform(X).equals(X_C))
print("D 일치:", transformer_D.transform(X).equals(X_D))

B 일치: True
C 일치: True
D 일치: True


### 3-2. 공통 모델 설정

In [23]:
# XGBoost 정상 작동 확인

import xgboost as xgb

print("XGBoost 버전:", xgb.__version__)

XGBoost 버전: 3.4.1


In [25]:
# 팀 공통 Baseline 설정으로 B·C·D Pipeline 생성

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

base_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

pipeline_B = Pipeline([
    ("features", transformer_B),
    ("model", base_model)
])

pipeline_C = Pipeline([
    ("features", transformer_C),
    ("model", base_model)
])

pipeline_D = Pipeline([
    ("features", transformer_D),
    ("model", base_model)
])

print("B:", pipeline_B.named_steps["model"].get_params()["n_estimators"])
print("C:", pipeline_C.named_steps["model"].get_params()["n_estimators"])
print("D:", pipeline_D.named_steps["model"].get_params()["n_estimators"])

B: 200
C: 200
D: 200


### 3-3. A·B·C·D 교차검증 성능 비교

In [28]:
y_encoded, label_encoder = encode_target(y)

In [29]:
# 타깃 인코딩 결과 확인

print("타깃 데이터 크기:", y_encoded.shape)
print("클래스 개수:", len(label_encoder.classes_))
print("인코딩 결과:", y_encoded[:10])

타깃 데이터 크기: (6201,)
클래스 개수: 26
인코딩 결과: [ 8 19 20  9  6 21  2 23 12 21]


#### A 교차검증 성능비교

In [33]:
...

Ellipsis

#### B 교차검증 성능비교

In [30]:
# B. 이진 인코딩 피처의 5-Fold 교차검증

from src.validation import get_folds, run_cv

folds = get_folds(y_encoded)

result_B = run_cv(
    model=pipeline_B,
    X=X,
    y=y_encoded,
    folds=folds
)

print(f"\nB 평균 Macro F1: {result_B['mean']:.4f}")
print(f"B 표준편차: {result_B['std']:.4f}")

fold 0: Macro F1 = 0.3160
fold 1: Macro F1 = 0.3194
fold 2: Macro F1 = 0.3412
fold 3: Macro F1 = 0.3089
fold 4: Macro F1 = 0.3146
CV Macro F1 = 0.3200 (std 0.0111)

B 평균 Macro F1: 0.3200
B 표준편차: 0.0111


#### C 교차검증 성능비교

In [31]:
# C. 이진 인코딩 + 변이 유전자 수 피처의 5-Fold 교차검증

result_C = run_cv(
    model=pipeline_C,
    X=X,
    y=y_encoded,
    folds=folds
)

print(f"\nC 평균 Macro F1: {result_C['mean']:.4f}")
print(f"C 표준편차: {result_C['std']:.4f}")

# B 대비 성능 변화
diff_C = result_C["mean"] - result_B["mean"]
print(f"B 대비 Macro F1 변화: {diff_C:+.4f}")

fold 0: Macro F1 = 0.3562
fold 1: Macro F1 = 0.3794
fold 2: Macro F1 = 0.3715
fold 3: Macro F1 = 0.3550
fold 4: Macro F1 = 0.3692
CV Macro F1 = 0.3663 (std 0.0093)

C 평균 Macro F1: 0.3663
C 표준편차: 0.0093
B 대비 Macro F1 변화: +0.0462


#### D 교차검증 성능비교

In [32]:
# D. 이진 인코딩 + 변이 유전자 수 + 변이 유형별 개수의 5-Fold 교차검증

result_D = run_cv(
    model=pipeline_D,
    X=X,
    y=y_encoded,
    folds=folds
)

print(f"\nD 평균 Macro F1: {result_D['mean']:.4f}")
print(f"D 표준편차: {result_D['std']:.4f}")

# B 및 C 대비 성능 변화
diff_D_B = result_D["mean"] - result_B["mean"]
diff_D_C = result_D["mean"] - result_C["mean"]

print(f"B 대비 Macro F1 변화: {diff_D_B:+.4f}")
print(f"C 대비 Macro F1 변화: {diff_D_C:+.4f}")

fold 0: Macro F1 = 0.3814
fold 1: Macro F1 = 0.3891
fold 2: Macro F1 = 0.3821
fold 3: Macro F1 = 0.3820
fold 4: Macro F1 = 0.3832
CV Macro F1 = 0.3836 (std 0.0028)

D 평균 Macro F1: 0.3836
D 표준편차: 0.0028
B 대비 Macro F1 변화: +0.0635
C 대비 Macro F1 변화: +0.0173


### 3-4. 실험 결과 기록 및 최종 피처 선정

In [34]:
# B·C·D 피처 구성별 교차검증 성능 비교표

import pandas as pd

comparison = pd.DataFrame({
    "피처 구성": [
        "B: 이진 인코딩",
        "C: 이진 + 변이 유전자 수",
        "D: 이진 + 변이 수 + 유형별 개수"
    ],
    "피처 수": [X_B.shape[1], X_C.shape[1], X_D.shape[1]],
    "Macro F1 평균": [
        result_B["mean"],
        result_C["mean"],
        result_D["mean"]
    ],
    "Macro F1 표준편차": [
        result_B["std"],
        result_C["std"],
        result_D["std"]
    ]
})

comparison["B 대비 개선"] = (
    comparison["Macro F1 평균"] - result_B["mean"]
)

display(comparison.round(4))

,피처 구성,피처 수,Macro F1 평균,Macro F1 표준편차,B 대비 개선
0,B: 이진 인코딩,4384,0.3200,0.0111,0.0000
1,C: 이진 + 변이 유전자 수,4385,0.3663,0.0093,0.0462
2,D: 이진 + 변이 수 + 유형별 개수,4390,0.3836,0.0028,0.0635


In [35]:
# B·C·D 피처 실험 결과를 팀 공통 형식으로 CSV에 기록

from src.evaluation import log_result

model_params = pipeline_B.named_steps["model"].get_params()

experiments = [
    (
        "B_binary",
        "4384개 유전자 이진 인코딩",
        result_B,
        "WT·결측=0, 변이=1"
    ),
    (
        "C_binary_mutation_count",
        "이진 인코딩 + n_mutated_genes",
        result_C,
        "환자별 변이 유전자 수 추가"
    ),
    (
        "D_binary_mutation_types",
        "이진 인코딩 + n_mutated_genes + 변이 유형별 개수 5개",
        result_D,
        "missense, synonymous, nonsense, frameshift, other 개수 추가"
    )
]

for experiment, features, cv, notes in experiments:
    log_result(
        owner="H_data",
        notebook="02_features.ipynb",
        experiment=experiment,
        features=features,
        model="XGBoost",
        params=model_params,
        cv=cv,
        notes=notes
    )

기록: log_H_data.csv ← B_binary (CV 0.3200)
기록: log_H_data.csv ← C_binary_mutation_count (CV 0.3663)
기록: log_H_data.csv ← D_binary_mutation_types (CV 0.3836)


In [36]:
# 실험 로그 저장 결과 확인

from src.config import EXPERIMENT_DIR

log_df = pd.read_csv(EXPERIMENT_DIR / "log_H_data.csv")

display(
    log_df[
        ["experiment", "features", "model", "cv_mean", "cv_std"]
    ]
)

,experiment,features,model,cv_mean,cv_std
0,B_binary,4384개 유전자 이진 인코딩,XGBoost,0.3200,0.0111
1,C_binary_mutation_count,이진 인코딩 + n_mutated_genes,XGBoost,0.3663,0.0093
2,D_binary_mutation_types,이진 인코딩 + n_mutated_genes + 변이 유형별 개수 5개,XGBoost,0.3836,0.0028


In [37]:
# src/features.py에 추가한 함수 정상 작동 확인

import importlib
import src.features

importlib.reload(src.features)

type_counts_shared = src.features.mutation_type_counts(X)

print("피처 크기:", type_counts_shared.shape)
print("기존 결과와 일치:", type_counts_shared.equals(type_counts))
print("결측치 개수:", type_counts_shared.isna().sum().sum())

피처 크기: (6201, 5)
기존 결과와 일치: True
결측치 개수: 0
